In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MENGABUNGKAN BATCH UTAMA DAN GEMPA MERUSAK
- Tanpa deduplikasi
- Mengabaikan file ._ (macOS metadata)
- Ekstrak timestamp dari berbagai format nama file
"""

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

DIR_MAIN = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
DIR_MAJOR = "/Volumes/Extreme SSD/unduhan_waveform_major_earthquakes"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_waveform_merged"
OUTPUT_CSV = "/Volumes/Extreme SSD/unduhan_waveform_merged/file_list_merged.csv"

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

def extract_timestamp(filename):
    """Ekstrak timestamp (YYYYMMDD) dari berbagai format nama file."""
    # Pola 1: BMKG-YYYYMMDD...
    match = re.search(r'BMKG-(\d{8})', filename)
    if match:
        return match.group(1)
    # Pola 2: officialYYYYMMDD...
    match = re.search(r'official(\d{8})', filename)
    if match:
        return match.group(1)
    # Pola 3: 8 digit angka di mana pun
    match = re.search(r'(\d{8})', filename)
    if match:
        return match.group(1)
    return None

def extract_full_timestamp(filename):
    """Ekstrak timestamp lengkap (YYYYMMDDHHMMSS) dari nama file."""
    # Pola 1: BMKG-YYYYMMDDHHMMSS
    match = re.search(r'BMKG-(\d{14})', filename)
    if match:
        return match.group(1)
    # Pola 2: officialYYYYMMDDHHMMSS
    match = re.search(r'official(\d{14})', filename)
    if match:
        return match.group(1)
    return None

print("="*70)
print("📊 MENGABUNGKAN DATASET UTAMA + GEMPA MERUSAK")
print("="*70)

# --- 1. Baca file ---
files_main = [f for f in Path(DIR_MAIN).glob("*.mseed") if not f.name.startswith('._')]
files_major = [f for f in Path(DIR_MAJOR).glob("*.mseed") if not f.name.startswith('._')]

print(f"\n✅ Batch utama: {len(files_main)} file")
print(f"✅ Batch gempa merusak: {len(files_major)} file")
print(f"✅ Total file gabungan: {len(files_main) + len(files_major)}")

# --- 2. Gabungkan dan ekstrak metadata ---
all_files = files_main + files_major
metadata = []

for f in all_files:
    parts = f.stem.split('_')
    if len(parts) >= 2:
        network = parts[0]
        station = parts[1]
    else:
        network = 'UNK'
        station = 'UNK'
    
    timestamp = extract_timestamp(f.name)
    full_timestamp = extract_full_timestamp(f.name)
    source = 'main' if DIR_MAIN in str(f.parent) else 'major'
    
    metadata.append({
        'file': f.name,
        'path': str(f),
        'network': network,
        'station': station,
        'timestamp': timestamp,
        'full_timestamp': full_timestamp,
        'source': source
    })

df = pd.DataFrame(metadata)
print(f"\n✅ Metadata diekstrak: {len(df)} file")

# --- 3. Cek duplikat ---
duplicates = df[df.duplicated(subset=['file'], keep=False)]
if len(duplicates) > 0:
    print(f"⚠️ Ditemukan {len(duplicates)} file dengan nama duplikat")
else:
    print("✅ Tidak ada file duplikat")

# --- 4. Statistik ---
print("\n" + "="*70)
print("📊 STATISTIK DATASET GABUNGAN")
print("="*70)

print(f"Total file unik: {len(df)}")
print(f"  - Batch utama: {len(df[df['source']=='main'])}")
print(f"  - Gempa merusak: {len(df[df['source']=='major'])}")

# Distribusi stasiun
print("\n📡 10 Stasiun Terbanyak:")
for sta, count in df['station'].value_counts().head(10).items():
    print(f"  {sta}: {count} ({count/len(df)*100:.1f}%)")

# Distribusi tahun
df['year'] = df['timestamp'].str[:4] if df['timestamp'].notna().any() else None
if df['year'].notna().any():
    yearly = df['year'].value_counts().sort_index()
    print("\n📊 Distribusi per Tahun:")
    for year, count in yearly.items():
        print(f"  {year}: {count}")

# --- 5. Simpan ---
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Daftar file gabungan: {OUTPUT_CSV}")

# --- 6. Opsi: Buat symbolic link (atau copy) ke folder output ---
print("\n💡 Untuk ekstraksi, Anda bisa:")
print(f"   1. Gunakan daftar file di: {OUTPUT_CSV}")
print(f"   2. Atau buat symbolic link:")
print(f"      mkdir -p {OUTPUT_DIR}")
print(f"      find {DIR_MAIN} -name '*.mseed' -exec ln -s {{}} {OUTPUT_DIR}/ \\;")
print(f"      find {DIR_MAJOR} -name '*.mseed' -exec ln -s {{}} {OUTPUT_DIR}/ \\;")

# --- 7. Visualisasi ---
try:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Pie chart sumber
    ax1 = axes[0, 0]
    source_counts = df['source'].value_counts()
    ax1.pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%')
    ax1.set_title('Sumber Data', fontsize=14)
    
    # Bar chart stasiun
    ax2 = axes[0, 1]
    top_stations = df['station'].value_counts().head(10)
    ax2.barh(top_stations.index, top_stations.values, color='steelblue', alpha=0.7)
    ax2.set_xlabel('Jumlah File')
    ax2.set_title('10 Stasiun Terbanyak', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    # Bar chart tahun
    ax3 = axes[1, 0]
    if df['year'].notna().any():
        yearly = df['year'].value_counts().sort_index()
        ax3.bar(yearly.index, yearly.values, color='coral', alpha=0.7)
        ax3.set_xlabel('Tahun')
        ax3.set_ylabel('Jumlah File')
        ax3.set_title('Distribusi per Tahun', fontsize=14)
        ax3.grid(True, alpha=0.3)
    
    # Summary table
    ax4 = axes[1, 1]
    ax4.axis('tight')
    ax4.axis('off')
    summary = [
        ['Total file unik', str(len(df))],
        ['Batch utama', str(len(df[df['source']=='main']))],
        ['Gempa merusak', str(len(df[df['source']=='major']))],
        ['Stasiun terbanyak', df['station'].value_counts().index[0] if len(df) > 0 else '-'],
        ['Rentang tahun', f"{df['year'].min()} - {df['year'].max()}" if df['year'].notna().any() else 'N/A'],
        ['Jumlah stasiun', str(df['station'].nunique())],
    ]
    table = ax4.table(cellText=summary, colLabels=['Metrik', 'Nilai'],
                      cellLoc='center', loc='center',
                      colColours=['#4472C4', '#4472C4'])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.8)
    ax4.set_title('Ringkasan Dataset Gabungan', fontsize=14, pad=20)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/merge_summary.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Grafik tersimpan: {OUTPUT_DIR}/merge_summary.png")
except Exception as e:
    print(f"⚠️ Visualisasi gagal: {e}")

print("\n" + "="*70)
print("📊 RINGKASAN")
print("="*70)
print(f"Total file unik: {len(df)}")
if df['year'].notna().any():
    print(f"Rentang tahun: {df['year'].min()} - {df['year'].max()}")
print(f"Jumlah stasiun unik: {df['station'].nunique()}")
print("="*70)